# 멀티카메라 비디오 처리 시스템 테스트

`videoSCST`를 사용한 멀티카메라 비디오 처리 테스트 (최적화 버전)


In [11]:
import os, sys, time, glob
import torch
sys.path.append("/workspace")

from MCMT_engine.SCST.video_SCST import videoSCST

## 1. 최적화된 Args 설정


In [12]:
class Args:
    # 기존 트래킹 파라미터 유지
    track_thresh = 0.3
    match_thresh = 0.9
    track_buffer = 180
    mot20 = False
    # 배치/파이프라인
    batch_size   = 1000   
    min_flush = 100        # 타임아웃을 쓰더라도 64장 모일 때만 플러시
    infer_timeout = 0.05
    cpu_workers  = 0       # 더 이상 사용 안 함(디코더가 스레드 내부에서 처리)
    chunk_sec    = 0.0     # 사용 안 함
    # 새 디코더 파라미터
    decode_threads  = 48    # CPU가 여유면 늘리고, 과하면 컨텍스트 스위칭 ↑ 0은 Auto
    prefetch_frames = 4000  # 여유 메모리 따라 128~512
    hwaccel         = "nvdec"  # "cuda"/"nvdec" 가능 시 사용
    decode_target_size = (1920, 1080)  # (1280,720) 처럼 지정하면 디코더에서 다운스케일


args = Args()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name()}")
else:
    print("CUDA를 사용할 수 없습니다. CPU 모드로 실행됩니다.")
    

# 1) 도면 이미지 경로 (공통)
plan_path = "/workspace/assets/seocho/Seocho_plan_pts.png"

GPU: NVIDIA RTX A6000


## 2. 비디오 파일 확인 및 공유 모델 초기화


In [16]:
# 4) 비디오 파일
video_paths = [
    "/workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #1.mp4",
    "/workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #2.mp4",
    "/workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #3.mp4",
]

existing_videos = []
for i, video_path in enumerate(video_paths):
    if os.path.exists(video_path):
        existing_videos.append(video_path)
        print(f"Camera {i+1}: {video_path}")
    else:
        print(f"Camera {i+1}: {video_path} (파일 없음)")

if len(existing_videos) < 3:
    raise FileNotFoundError(f"비디오 3개가 모두 필요합니다. 현재 {len(existing_videos)}개만 확인됨.")

print(f"\n총 {len(existing_videos)}개 비디오 파일 확인됨")


# 5) 결과 저장 디렉토리
os.makedirs("/workspace/results", exist_ok=True)
print("결과 저장 디렉토리 준비 완료")


Camera 1: /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #1.mp4
Camera 2: /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #2.mp4
Camera 3: /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #3.mp4

총 3개 비디오 파일 확인됨
결과 저장 디렉토리 준비 완료


In [17]:
# 6) videoSCST 초기화
print("\nvideoSCST 초기화 중...")
scst = videoSCST(
    plan_path=plan_path,
    args=args,
    det_models=["ultra_best"],     # 실제 모델 키 확인 필요
    det_device="cuda:0",
    det_threshold=0.0,
    det_use_async=True,
    det_max_workers=1,
)
print("videoSCST 초기화 완료")


videoSCST 초기화 중...
[SCST] __init__: plan_path=/workspace/assets/seocho/Seocho_plan_pts.png
[SCST] __init__: create DetectionAPI(device=cuda:0, models=['ultra_best'], async=True)
DetectionAPI activate
[DEBUG] VehicleDetector loaded with Ultralytics YOLO on cuda:0
[SCST] __init__: create TrackerAPI (thin)
videoSCST 초기화 완료


In [18]:
# 7) 각 카메라별 처리
start_time = time.time()
results = []

print("캘리브레이션 포인트 설정 완료")

캘리브레이션 포인트 설정 완료


In [ ]:
from assets.cord.cam1_plan_downsized import CAM1_PTS, CAM1_PLAN_PTS

# Camera 1
print("\nCamera 1 처리 시작...")
cam_start = time.time()
result1 = scst.track_and_save(
    video_path="/workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #1.mp4",
    cam_pts=CAM1_PTS,
    plan_pts=CAM1_PLAN_PTS,
    plan_img_path="/workspace/assets/seocho/plan.png",
    camera_save_path="/workspace/results/tracking_result1new.mp4",
    plan_save_path="/workspace/results/plan_result1.mp4",
    cam_trail_len=30,
    plan_stride=1,          # 필요 시 2~3으로 올려 저장 부하/용량 감소
    inplace_clear=True      # 메모리 즉시 해제
)
results.append(result1)
cam_time = time.time() - cam_start
print(f"Camera 1 처리 완료 ({cam_time:.2f}초, {len(result1)} 프레임)")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Camera 1 처리 시작...
[SCST] calibrate: start
[SCST] _ensure_projector: assign plan image (/workspace/assets/seocho/2025-09-09_seocho.png)
[SCST] calibrate: create PlanProjector + initial H fit
[SCST] calibrate: done |H|=4304.0375 time=1.223s
[SCST] _get_fps: read fps from /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #1.mp4
[SCST] plan-encode(online): open writer /workspace/results/plan_result1.mp4 (fps=30.00245504214973)
[SCST] tracking: begin (streaming decoder → detect & associate in TrackerCore)
[SCST] _get_fps: read fps from /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #1.mp4
[decoder] open(PyAV): threads=48, hwaccel=nvdec
DetectionAPI : Batch Detection..


KeyboardInterrupt: 

In [ ]:
from assets.cord.cam2_plan_downsized import CAM2_PTS, CAM2_PLAN_PTS

# Camera 2
print("\nCamera 2 처리 시작...")
cam_start = time.time()
result2 = scst.track_and_save(
    video_path="/workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #2.mp4",
    cam_pts=CAM2_PTS,
    plan_pts=CAM2_PLAN_PTS,
    plan_img_path="/workspace/assets/seocho/plan.png",
    camera_save_path="/workspace/results/tracking_result2new.mp4",
    plan_save_path="/workspace/results/plan_result2.mp4",
    cam_trail_len=30,
    plan_stride=1,
    inplace_clear=True
)
results.append(result2)
cam_time = time.time() - cam_start
print(f"Camera 2 처리 완료 ({cam_time:.2f}초, {len(result2)} 프레임)")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Camera 2 처리 시작...
[SCST] calibrate: start
[SCST] calibrate: refit homography with new points
[SCST] calibrate: done |H|=1244.7006 time=0.020s
[SCST] _get_fps: read fps from /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #2.mp4
[SCST] plan-encode(online): open writer /workspace/results/plan_result2.mp4 (fps=29.916315935618623)
[SCST] tracking: begin (streaming decoder → detect & associate in TrackerCore)
[SCST] _get_fps: read fps from /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #2.mp4
[decoder] open(PyAV): threads=48, hwaccel=nvdec
DetectionAPI : Batch Detection..


KeyboardInterrupt: 

[SCST] plan-encode(online): progress 100 frames written


In [ ]:
from assets.cord.cam3_plan_downsized import CAM3_PTS, CAM3_PLAN_PTS

# Camera 3
print("\nCamera 3 처리 시작...")
cam_start = time.time()
result3 = scst.track_and_save(
    video_path="/workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #3.mp4",
    cam_pts=CAM3_PTS,
    plan_pts=CAM3_PLAN_PTS,
    plan_img_path="/workspace/assets/seocho/plan.png",
    camera_save_path="/workspace/results/tracking_result3new.mp4",
    plan_save_path="/workspace/results/plan_result3.mp4",
    cam_trail_len=30,
    plan_stride=1,
    inplace_clear=True
)
results.append(result3)
cam_time = time.time() - cam_start
print(f"Camera 3 처리 완료 ({cam_time:.2f}초, {len(result3)} 프레임)")
if torch.cuda.is_available():
    torch.cuda.empty_cache()


Camera 3 처리 시작...
[SCST] calibrate: start
[SCST] calibrate: refit homography with new points
[SCST] calibrate: done |H|=7015.3491 time=0.040s
[SCST] _get_fps: read fps from /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #3.mp4
[SCST] plan-encode(online): open writer /workspace/results/plan_result3.mp4 (fps=29.98667449350574)
[SCST] tracking: begin (streaming decoder → detect & associate in TrackerCore)
[SCST] _get_fps: read fps from /workspace/datasets/250909_Site_seocho/2025-09-09 13_29_59 이동형 #3.mp4
[decoder] open(PyAV): threads=48, hwaccel=nvdec
DetectionAPI : Batch Detection..
DetectionAPI : Batch Detection..
DetectionAPI : Batch Detection..
DetectionAPI : Batch Detection..


In [ ]:
# 8) 요약
total_time = time.time() - start_time
total_frames = sum(len(r) for r in results)
print(f"\n전체 완료: {total_time:.2f}초, 총 {total_frames} 프레임")

## 4. 결과 요약 및 리소스 정리


In [ ]:
# 9) 결과 요약 및 파일 확인
print("\n모든 카메라 처리 완료!")
print(f"총 처리 시간: {total_time:.2f}초")
print(f"총 처리 프레임: {total_frames:,}개")
print(f"평균 처리 속도: {total_frames/total_time:.1f} FPS")
print("결과 저장 위치: /workspace/results/")

result_files = glob.glob("/workspace/results/*.mp4")
print(f"\n생성된 결과 파일 ({len(result_files)}개):")
for fp in sorted(result_files):
    size_mb = os.path.getsize(fp) / (1024*1024)
    print(f"  - {os.path.basename(fp)}  ({size_mb:.1f} MB)")

print("\n완료")

In [ ]:
scst.close()